Document ingestion


In [1]:
!pip install -q langchain langchain-community sentence-transformers faiss-cpu pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [1]:
from google.colab import files
uploaded = files.upload()

Saving iForm Custom Coding Guideline 1 - Copy (1).pdf to iForm Custom Coding Guideline 1 - Copy (1) (1).pdf


In [2]:
file_path=list(uploaded.keys())[0]
print("uploaded file:",file_path)


uploaded file: iForm Custom Coding Guideline 1 - Copy (1) (1).pdf


In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
loader=PyMuPDFLoader(file_path)
documents=loader.load()
print("number of pages loaded:",len(documents))


number of pages loaded: 65


In [4]:
import re
def clean_text(text):
  text=re.sub(r'\n+','\n',text)
  text=re.sub(r'\s+','',text)
  text = re.sub(r'\.{2,}', '.', text)
  text = text.strip()
  return text

for doc in documents:
    doc.page_content = clean_text(doc.page_content)

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(documents)

print("Total chunks generated:", len(chunks))

Total chunks generated: 125


In [6]:
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["source_document"] = file_path

In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:151: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)

In [9]:
vector_db.save_local("vector_store")

In [10]:
vector_db = FAISS.load_local(
    "vector_store",
    embedding_model,
    allow_dangerous_deserialization=True
)

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

In [11]:
!pip install -q openai

In [12]:
import os
os.environ["OPENROUTER_API_KEY"] = "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"

In [13]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [14]:
full_document_text = " ".join([doc.page_content for doc in documents])

In [16]:
!pip install \
langchain==0.2.14 \
langchain-core==0.2.35 \
langchain-community==0.2.12 \
langchain-openai==0.1.22

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.8/997.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.9/394.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.4
    Uninstalling tenacity-9.1.4:
      Succ

In [15]:
from langchain_openai import ChatOpenAI
import os

MODEL_NAME = "meta-llama/llama-3-8b-instruct"

llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.3
)

In [16]:
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

In [17]:
import langchain
print(langchain.__version__)

0.2.14


In [18]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [19]:
def rag_tool(query):

    docs = vector_db.similarity_search(query, k=4)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt)

    citations = []

    for doc in docs:
        snippet = doc.page_content[:150]
        chunk_id = doc.metadata["chunk_id"]

        citations.append(f"[Chunk {chunk_id}] {snippet}...")

    citation_text = "\n\n".join(citations)

    return f"""
{response.content}

Sources:
{citation_text}
"""

In [20]:
rag_tool("What is this document about?")

'\nThis document appears to be a set of guidelines for custom coding in iForm, a mobile form-building platform. It provides information on various functions and methods that can be used to customize the behavior and appearance of forms, including post-save hooks, table operations, and control value retrieval. It also covers topics such as pattern validation for text controls and formatting for currency and number values.\n\nSources:\n[Chunk 13] iFormCustomCodingGuidelinePage7...\n\n[Chunk 115] .117PostHookonSaveofWorkitemPurpose:Thisiscalledafterworkitemissaved.Signature:saveWorkItemPostHook()Returnvalue:noneExample:definethefunction:iFormCu...\n\n[Chunk 5] .182.19SetaParticularCellDisabled/EnabledinaTable.182.20ClearDataofTable.182.21SaveWorkItem.192.22CompleteWorkItem.192.23GetWorkItemData.192.24SetRowS...\n\n[Chunk 76] .controlId:controlIdofthetextControl2.pattern:newpatternforthetextcontrolForpattern,youhavetoprovidethevalueasgivenbelow:TypeofPatternValuetobepassedi...\n'

In [21]:
def summarize_tool(query: str):

    prompt = f"""
Provide a clear summary of the following document:

{full_document_text[:12000]}
"""

    return llm.invoke(prompt).content

In [22]:
def critique_tool(query: str):

    prompt = f"""
Critically evaluate the following document.

Provide:

• Strengths
• Weaknesses
• Potential bias
• Missing perspectives

Document:

{full_document_text[:12000]}
"""

    return llm.invoke(prompt).content

In [23]:
from langchain.tools import Tool

tools = [

    Tool(
        name="Document_Summarizer",
        func=summarize_tool,
        description="Use this tool when the user asks for a summary or overview of the document."
    ),

    Tool(
        name="Document_QA",
        func=rag_tool,
        description="Use this tool when the user asks specific questions about the document."
    ),

    Tool(
        name="Document_Critique",
        func=critique_tool,
        description="Use this tool when the user asks to critique, analyze, or evaluate the document."
    )

]

In [24]:
def classify_query(query):

    prompt = f"""
You are an AI assistant that routes queries to tools.

Choose ONE tool for the query.

TOOLS:

1. summarizer
Use when the user asks for an overview of the entire document.

2. qa
Use when the user asks specific questions about a topic in the document.

3. critique
Use when the user asks for evaluation, strengths, weaknesses, bias, or analysis.

Return ONLY the tool name.

Query: {query}
"""

    response = llm.invoke(prompt)

    return response.content.strip().lower()

In [25]:
def agent_router(query):

    q = query.lower()

    # clear summary intent
    if any(word in q for word in [
        "summary","summarize","overview"
    ]):
        return "Summarizer Tool", summarize_tool(query)

    # critique intent
    if any(word in q for word in [
        "critique","bias","limitations","weakness"
    ]):
        return "Critique Tool", critique_tool(query)

    # otherwise let LLM decide
    tool = classify_query(query)

    if "summarizer" in tool:
        return "Summarizer Tool", summarize_tool(query)

    if "critique" in tool:
        return "Critique Tool", critique_tool(query)

    return "General Q&A Tool", rag_tool(query)

In [26]:
from langchain.agents import initialize_agent
from langchain.agents import AgentType

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:151: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


In [27]:
def ask_agent(query):

    response = agent.run(query)

    return response

In [28]:
def process_query(query):

    if vector_db is None:
        return "⚠️ Please upload a document first."

    tool, response = agent_router(query)

    return f"""
🔧 Tool Used: {tool}

-----------------------------

{response}
"""

In [29]:
def ingest_document(file):

    global vector_db
    global documents
    global full_document_text

    file_path = file.name   # path from Gradio upload

    loader = PyMuPDFLoader(file_path)
    documents = loader.load()

    chunks = text_splitter.split_documents(documents)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i

    vector_db = FAISS.from_documents(chunks, embedding_model)

    full_document_text = " ".join([doc.page_content for doc in documents])

    return "✅ Document processed successfully!"

In [32]:
import gradio as gr

In [34]:
with gr.Blocks() as demo:

    gr.Markdown("# 📄 AI Document Assistant")

    gr.Markdown("""
Upload a PDF and ask questions about it.

### Example queries

Document overview:
• Give me an overview of the document
• Summarize this report

Specific questions:
• What does the document say about renewable energy?
• Explain the section on climate change

Critical analysis:
• Critique this document
• What are the weaknesses of this report?
""")

    file = gr.File(label="Upload PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        fn=ingest_document,
        inputs=file,
        outputs=status
    )

    query = gr.Textbox(
        label="Ask a question about the document"
    )

    ask_button = gr.Button("Ask")

    output = gr.Textbox(label="AI Response", lines=10)

    ask_button.click(
        fn=process_query,
        inputs=query,
        outputs=output
    )

demo.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0835e94aa28fdfa79.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
